# Classification automatique de décisions judiciaires françaises par type d'infraction
**Awa Traoré DIOP | Projet NLP personnel | Data Science**

## Mise à jour — Août 2026

Ce notebook intègre deux évolutions majeures de l'API Judilibre :

- **`pyjudilibre`** : librairie Python officielle (juin 2026) — remplace les appels `requests` manuels
- **Décisions pénales disponibles** : depuis le 31 décembre 2025, les décisions criminelles et correctionnelles des cours d'appel sont accessibles — bien plus pertinentes pour la classification par type d'infraction

---
## Vue d'ensemble du projet

Ce projet combine deux sources de données publiques françaises :

- **Source 1 — Labels** : Les catégories d'infractions de l'ONDRP (data.gouv.fr)
- **Source 2 — Textes** : Décisions de justice via l'API Judilibre (Cour de cassation + Cours d'appel)

**Objectif** : Classifier automatiquement une décision de justice par type d'infraction

**Application directe** : détection de fraudes (DGDDI), analyse de dossiers médicaux (santé)

---
## Prérequis

**Clé API Judilibre** (gratuit) : https://piste.api.gouv.fr
→ Section **API Keys** de l'application PISTE

**GPU Colab** : Exécution → Modifier le type d'exécution → GPU T4

---
## Phase 0 — Installation & Configuration
---

In [ ]:
# Nouveauté : pyjudilibre remplace les appels requests manuels
# Plus simple, plus robuste, maintenu par la Cour de cassation
!pip install pyjudilibre transformers datasets spacy torch scikit-learn -q
!python -m spacy download fr_core_news_md -q
print('Installation terminee !')

In [ ]:
import os
os.kill(os.getpid(), 9)  # Redémarre le kernel automatiquementimport os

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re, json, time
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Imports OK')

In [ ]:
from pyjudilibre import JudilibreClient
import os

API_KEY = '61097a32-d929-41ad-9e32-060529eb3801'

# Provide the API key and API URL explicitly
# Using the API URL provided by the user
client = JudilibreClient(judilibre_api_key=API_KEY, judilibre_api_url='https://api.piste.gouv.fr/cassation/judilibre/v1.0')

try:
    health = client.healthcheck()
    print(f'Connexion API Judilibre OK : {health}')
except Exception as e:
    print(f'Erreur de connexion : {e}')
    print('Verifie ta cle API dans la section API Keys de PISTE')

---
## Phase 1 — Construction des labels depuis l'ONDRP
---

Les 107 catégories d'infractions de l'ONDRP servent de référentiel de labels.
On les regroupe en 6 grandes familles pour avoir des classes équilibrées.

In [ ]:
import requests

# Téléchargement du fichier ONDRP depuis data.gouv.fr
# Note : les 107 catégories (Etat 4001) sont stables depuis 1972
# On les utilise uniquement pour les libellés, pas les volumes statistiques
url_ondrp = 'https://www.data.gouv.fr/api/1/datasets/r/0dc8dca0-b608-49fc-9dbc-cfa7fd5a221a'
print('Téléchargement du fichier ONDRP...')

response = requests.get(url_ondrp)
with open('ondrp_infractions.xlsx', 'wb') as f:
    f.write(response.content)

import pandas as pd
xl = pd.ExcelFile('ondrp_infractions.xlsx')
print(f'Feuilles disponibles : {xl.sheet_names}')

In [ ]:
# Lecture et exploration de la structure
df_raw = pd.read_excel('ondrp_infractions.xlsx', sheet_name=0, header=None)
print(f'Dimensions : {df_raw.shape}')
df_raw.head(20)

In [ ]:
# Extraction des libellés d'infractions
# Adapte COL_INFRACTIONS selon la cellule précédente
COL_INFRACTIONS = 2

infractions_raw = df_raw[COL_INFRACTIONS].dropna().tolist()
infractions = [str(i).strip() for i in infractions_raw if len(str(i).strip()) > 5]

print(f'Infractions extraites : {len(infractions)}')
for i, inf in enumerate(infractions[:15], 1):
    print(f'{i:3}. {inf}')

In [ ]:
# Regroupement en 6 catégories thématiques
CATEGORIES = {
    'Atteintes aux personnes': [
        'homicide', 'meurtre', 'coups', 'blessures', 'violence', 'viol',
        'agression', 'menace', 'harcelement', 'harcèlement', 'personne'
    ],
    'Atteintes aux biens': [
        'vol', 'cambriolage', 'escroquerie', 'fraude', 'recel', 'destruction',
        'degradation', 'dégradation', 'bien', 'propriete', 'propriété'
    ],
    'Infractions economiques': [
        'blanchiment', 'corruption', 'trafic', 'contrefacon', 'contrefaçon',
        'faux', 'abus', 'detournement', 'détournement', 'douane'
    ],
    'Infractions aux personnes vulnerables': [
        'enfant', 'mineur', 'famille', 'abandon', 'maltraitance', 'proxenetisme',
        'proxénétisme', 'traite'
    ],
    'Infractions a l ordre public': [
        'stupefiant', 'stupéfiant', 'arme', 'terrorisme', 'association',
        'rebellion', 'rébellion', 'outrage'
    ],
    'Infractions routieres': [
        'conduite', 'alcool', 'vehicule', 'véhicule', 'route', 'accident'
    ]
}

def categoriser(infraction):
    inf_lower = infraction.lower()
    for categorie, mots_cles in CATEGORIES.items():
        if any(mot in inf_lower for mot in mots_cles):
            return categorie
    return 'Autres infractions'

df_labels = pd.DataFrame({'infraction': infractions})
df_labels['categorie'] = df_labels['infraction'].apply(categoriser)

label2id = {label: idx for idx, label in enumerate(df_labels['categorie'].unique())}
id2label = {v: k for k, v in label2id.items()}

print('Distribution des categories :')
print(df_labels['categorie'].value_counts())
print(f'\nMapping : {label2id}')

---
## Phase 2 — Collecte des décisions via pyjudilibre
---

### Mise à jour majeure — Décisions pénales disponibles

Depuis le 31 décembre 2025, les décisions des cours d'appel en matière **criminelle et correctionnelle** sont disponibles dans Judilibre.

Pour le projet de classification par type d'infraction, c'est un changement fondamental :
- Avant : surtout des décisions civiles et commerciales
- Maintenant : décisions pénales réelles, directement liées aux catégories d'infractions

On utilise le paramètre `type` de l'API pour cibler les décisions pénales.

### Avantages de pyjudilibre vs requests manuel

- Validation automatique des paramètres
- Gestion des erreurs intégrée
- Code 3x plus court et plus lisible

In [ ]:
def rechercher_decisions_v2(mot_cle, batch_size=10, page=0):
    """
    Version mise à jour avec pyjudilibre
    Cible les décisions pénales (criminelles + correctionnelles)
    disponibles depuis le 31/12/2025
    """
    try:
        # pyjudilibre gère automatiquement les headers et l'auth
        resultats = client.search(
            query=mot_cle,
            page_size=batch_size,
            page=page,
            # Nouveauté : cibler les décisions pénales
            # 'arret' pour les cours d'appel, 'other' pour les autres
            # Laisse vide pour tout récupérer
        )

        # Debugging: Print resultats to inspect its structure
        print(f"Type of resultats: {type(resultats)}")
        print(f"Content of resultats: {resultats}")

        # The actual search results are in the second element of the tuple
        decisions = []
        for r in resultats[1]:
            texte = getattr(r, 'text', '') or getattr(r, 'summary', '') or ''
            if texte and len(texte.split()) > 30:
                decisions.append({
                    'id'      : getattr(r, 'id', ''),
                    'date'    : getattr(r, 'decision_date', ''),
                    'chambre' : getattr(r, 'chamber', ''),
                    'jurisdiction': getattr(r, 'jurisdiction', ''),
                    'texte'   : texte[:2000],
                    'mot_cle' : mot_cle
                })
        return decisions

    except Exception as e:
        print(f'  Erreur pour {mot_cle} : {e}')
        return []

# Test
test = rechercher_decisions_v2('vol', batch_size=2)
if test:
    print(f'Test OK : {len(test)} decision(s)')
    print(f'Juridiction : {test[0]["jurisdiction"]}')
    print(f'Extrait : {test[0]["texte"][:200]}...')
else:
    print('Test echoue — verifie ta cle API')

In [ ]:
# Alternative : utiliser l'endpoint /export pour récupérer en masse
# Utile si on veut constituer un corpus plus large
# Disponible dans pyjudilibre via client.export()

def export_decisions_batch(mot_cle, batch_size=20, page=0):
    """
    Utilise l'endpoint export de Judilibre
    Permet de récupérer des lots de décisions complètes
    """
    try:
        resultats = client.export(
            query=mot_cle,
            batch_size=batch_size,
            batch=page
        )
        decisions = []
        for r in resultats.results:
            texte = getattr(r, 'text', '') or ''
            if texte and len(texte.split()) > 50:
                decisions.append({
                    'id'      : getattr(r, 'id', ''),
                    'date'    : getattr(r, 'decision_date', ''),
                    'chambre' : getattr(r, 'chamber', ''),
                    'jurisdiction': getattr(r, 'jurisdiction', ''),
                    'texte'   : texte[:2000],
                    'mot_cle' : mot_cle
                })
        return decisions
    except Exception as e:
        print(f'  Export erreur pour {mot_cle} : {e}')
        # Fallback sur search si export non disponible
        return rechercher_decisions_v2(mot_cle, batch_size, page)

print('Fonctions de collecte pretes')

In [ ]:
# Collecte complète : 50 décisions par catégorie
REQUETES_PAR_CATEGORIE = {
    'Atteintes aux personnes'               : ['violence', 'coups blessures', 'agression'],
    'Atteintes aux biens'                   : ['vol', 'escroquerie', 'cambriolage'],
    'Infractions economiques'               : ['fraude', 'blanchiment', 'corruption'],
    'Infractions aux personnes vulnerables' : ['mineur', 'abandon famille'],
    'Infractions a l ordre public'          : ['stupefiant', 'arme', 'terrorisme'],
    'Infractions routieres'                 : ['conduite alcool', 'accident vehicule'],
}

NB_PAR_CATEGORIE = 50
corpus = []

for categorie, requetes in REQUETES_PAR_CATEGORIE.items():
    print(f'\nCategorie : {categorie}')
    decisions_categorie = []

    for requete in requetes:
        if len(decisions_categorie) >= NB_PAR_CATEGORIE:
            break
        restant = NB_PAR_CATEGORIE - len(decisions_categorie)
        # Essai export d'abord, fallback sur search
        decisions = export_decisions_batch(requete, batch_size=min(restant, 20))
        for d in decisions:
            d['categorie'] = categorie
            d['label'] = label2id.get(categorie, 6)
        decisions_categorie.extend(decisions)
        time.sleep(0.5)  # respect des quotas API

    corpus.extend(decisions_categorie)
    print(f'  -> {len(decisions_categorie)} decisions collectees')

df_corpus = pd.DataFrame(corpus)
print(f'\nCorpus total : {len(df_corpus)} decisions')
print(df_corpus['categorie'].value_counts())

# Sauvegarde
df_corpus.to_csv('corpus_judilibre_v2.csv', index=False, encoding='utf-8')
print('\nCorpus sauvegarde dans corpus_judilibre_v2.csv')

In [ ]:
# Exploration du corpus
df_corpus = pd.read_csv('corpus_judilibre_v2.csv')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution des catégories
counts = df_corpus['categorie'].value_counts()
colors_bar = plt.cm.Blues(np.linspace(0.4, 0.9, len(counts)))
axes[0].barh(counts.index, counts.values, color=colors_bar)
axes[0].set_title('Decisions par categorie', fontsize=12, fontweight='bold')
for i, v in enumerate(counts.values):
    axes[0].text(v + 0.3, i, str(v), va='center')

# Longueur des textes
df_corpus['nb_mots'] = df_corpus['texte'].apply(lambda x: len(str(x).split()))
axes[1].hist(df_corpus['nb_mots'], bins=30, color='#2E86AB', alpha=0.8)
axes[1].set_title('Longueur des decisions', fontsize=12, fontweight='bold')
axes[1].axvline(df_corpus['nb_mots'].median(), color='red', linestyle='--',
                label=f'Mediane : {df_corpus["nb_mots"].median():.0f}')
axes[1].legend()

# Nouveauté : distribution par juridiction
if 'jurisdiction' in df_corpus.columns:
    jurid_counts = df_corpus['jurisdiction'].value_counts().head(8)
    axes[2].bar(jurid_counts.index, jurid_counts.values, color='#2E86AB')
    axes[2].set_title('Decisions par juridiction', fontsize=12, fontweight='bold')
    axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('exploration_corpus_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nExemple de decision :')
ex = df_corpus.iloc[0]
print(f'Categorie    : {ex["categorie"]}')
print(f'Juridiction  : {ex.get("jurisdiction", "N/A")}')
print(f'Chambre      : {ex["chambre"]}')
print(f'Texte        : {str(ex["texte"])[:300]}...')

---
## Phase 3 — Modélisation NLP avec CamemBERT
---

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

df_model = df_corpus[['texte', 'categorie', 'label']].dropna()
df_model['texte'] = df_model['texte'].astype(str)

# Get only the categories actually present in the corpus
actual_categories_in_corpus = sorted(df_model['categorie'].unique())

# Create new label2id and id2label mappings for the present categories
# This ensures labels are 0-indexed and contiguous (e.g., 0, 1, 2, ...)
# Correction : commencer l'indexation à 0 pour être compatible avec les modèles Hugging Face
new_label2id = {label: idx for idx, label in enumerate(actual_categories_in_corpus, start=0)}
new_id2label = {v: k for k, v in new_label2id.items()}

# Remap the 'label' column in df_model using the new contiguous labels
df_model['label'] = df_model['categorie'].map(new_label2id)

# Labels présents dans les données (now these are 0-indexed and contiguous)
labels_presents = sorted(df_model['label'].unique())
categories_presentes = [new_id2label[int(i)] for i in labels_presents] # Use new_id2label here

X_train, X_test, y_train, y_test = train_test_split(
    df_model['texte'], df_model['label'],
    test_size=0.2, random_state=42,
    stratify=df_model['label'] # Stratification pour maintenir la distribution des labels
)

print(f'Train : {len(X_train)} decisions')
print(f'Test  : {len(X_test)} decisions')
print(f'Categories presentes : {categories_presentes}')

# --- Vérification explicite de la présence de tous les labels dans les deux ensembles ---
all_original_labels = set(df_model['label'].unique())
labels_in_y_train = set(y_train.unique())
labels_in_y_test = set(y_test.unique())

missing_in_train = all_original_labels - labels_in_y_train
missing_in_test = all_original_labels - labels_in_y_test

if not missing_in_train and not missing_in_test:
    print("\nConfirmation : Tous les labels sont présents dans les ensembles d'entraînement et de test.")
else:
    if missing_in_train:
        print(f"\nATTENTION : Labels manquants dans le jeu d'entraînement : {missing_in_train}")
    if missing_in_test:
        print(f"\nATTENTION : Labels manquants dans le jeu de test : {missing_in_test}")
    print("\nCeci peut indiquer que certaines classes ont un nombre d'échantillons très faible pour la stratification.")
# ----------------------------------------------------------------------------------------

# Update the global id2label and label2id for subsequent cells (like c18 and c20)
id2label = new_id2label
label2id = new_label2id

In [ ]:
# Baseline TF-IDF
print('Baseline TF-IDF + Logistic Regression...')
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=500)
lr.fit(X_train_tfidf, y_train)
y_pred_baseline = lr.predict(X_test_tfidf)

print('\nBaseline :')
print('=' * 60)
print(classification_report(
    y_test, y_pred_baseline,
    labels=labels_presents,
    target_names=categories_presentes
))

In [ ]:
# ── MODELE PRINCIPAL : CamemBERT ──
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from transformers import CamembertTokenizer
from transformers import AutoTokenizer
from datasets import Dataset


MODEL_NAME = 'camembert-base'
NUM_LABELS = len(label2id)

# Tokenisation
print('Chargement du tokenizer CamemBERT...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=256  # decisions plus longues qu'Allocine -> 256 tokens
    )

# Construction des datasets HuggingFace
train_df = pd.DataFrame({'text': X_train.tolist(), 'label': y_train.tolist()})
test_df  = pd.DataFrame({'text': X_test.tolist(),  'label': y_test.tolist()})

train_dataset = Dataset.from_pandas(train_df).map(tokenize, batched=True)
test_dataset  = Dataset.from_pandas(test_df).map(tokenize,  batched=True)

print(f'Dataset train tokenise : {len(train_dataset)} exemples')
print(f'Dataset test tokenise  : {len(test_dataset)} exemples')

In [ ]:
# Chargement du modèle
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results_judilibre2",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",        # ← renommé en 5.x (était evaluation_strategy)
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    weight_decay=0.01,
    report_to="none",
    fp16=torch.cuda.is_available() # ← active la précision mixte si GPU dispo
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

if torch.cuda.is_available():
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
else:
    print("⚠ Aucun GPU — entraînement lent. Active le GPU dans Exécution → Modifier le type d'exécution")

print("Entraînement CamemBERT lancé...")
trainer.train()
print("✅ Entraînement terminé !")

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
!pip show transformers

---
## Phase 4 — Evaluation et comparaison des modeles
---

In [ ]:
from sklearn.metrics import f1_score

predictions = trainer.predict(test_dataset)
y_pred_camembert = np.argmax(predictions.predictions, axis=1)
y_true = test_dataset['label']

print('CamemBERT fine-tune :')
print('=' * 60)
print(classification_report(
    y_true, y_pred_camembert,
    labels=labels_presents,
    target_names=categories_presentes
))

f1_baseline  = f1_score(y_test, y_pred_baseline,   average='weighted')
f1_camembert = f1_score(y_true, y_pred_camembert,  average='weighted')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Comparaison F1
bars = axes[0].bar(['TF-IDF\n+LogReg', 'CamemBERT\nfine-tune'],
                   [f1_baseline, f1_camembert],
                   color=['#ADB5BD', '#2E86AB'], width=0.5)
axes[0].set_ylim(0, 1)
axes[0].set_title('F1-score weighted', fontsize=12, fontweight='bold')
for bar, score in zip(bars, [f1_baseline, f1_camembert]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{score:.3f}', ha='center', fontweight='bold')

# Matrices de confusion
short_cats = [c[:15] for c in categories_presentes]
for ax, y_p, title in [
    (axes[1], y_pred_baseline,  'Baseline'),
    (axes[2], y_pred_camembert, 'CamemBERT')
]:
    cm = confusion_matrix(y_true if title == 'CamemBERT' else y_test, y_p, labels=labels_presents)
    ConfusionMatrixDisplay(cm, display_labels=short_cats).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Confusion — {title}', fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('comparaison_modeles_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Gain CamemBERT vs Baseline : +{(f1_camembert-f1_baseline)*100:.1f} pts F1')

---
## Phase 5 — Extraction d'entités nommées avec spaCy
---

In [ ]:
import spacy
nlp = spacy.load('fr_core_news_md')

def extraire_entites(texte):
    doc = nlp(texte[:1000])
    return [(ent.text, ent.label_) for ent in doc.ents]

# Analyse sur un échantillon par catégorie
sample = df_corpus.groupby('categorie').first().reset_index()
print('Entites par categorie :')
print('=' * 60)
for _, row in sample.iterrows():
    entites = extraire_entites(str(row['texte']))
    print(f'\n{row["categorie"]}')
    for texte_e, type_e in entites[:5]:
        print(f'  [{type_e}] {texte_e}')

In [ ]:
# Analyse globale des entités
toutes_entites = []
for texte in df_corpus['texte'].head(100):
    toutes_entites.extend(extraire_entites(str(texte)))

df_entites = pd.DataFrame(toutes_entites, columns=['texte', 'type'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
type_counts = df_entites['type'].value_counts().head(10)
axes[0].bar(type_counts.index, type_counts.values, color='#2E86AB')
axes[0].set_title('Types d entites les plus frequents', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

top_entites = df_entites['texte'].value_counts().head(15)
axes[1].barh(top_entites.index, top_entites.values, color='#2E86AB')
axes[1].set_title('Entites les plus mentionnees', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('ner_analyse_v2.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 6 — Demo pipeline complet
---

In [ ]:
def pipeline_complet(texte_decision):
    print('ANALYSE DE LA DECISION')
    print('=' * 60)
    print(f'Texte : {texte_decision[:200]}...')
    print()

    # Classification
    inputs = tokenizer(texte_decision, return_tensors='pt', truncation=True, max_length=256)
    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0]
    pred_id = torch.argmax(probs).item()
    pred_label = id2label.get(pred_id, 'Inconnu')
    confidence = probs[pred_id].item()

    print(f'CLASSIFICATION')
    print(f'  Categorie : {pred_label} ({confidence:.1%})')
    print(f'  Top 3 :')
    top3 = torch.topk(probs, min(3, len(probs)))
    for score, idx in zip(top3.values, top3.indices):
        cat = id2label.get(idx.item(), 'Inconnu')
        print(f'    {cat:<45} {score.item():.1%}')
    print()

    # NER
    entites = extraire_entites(texte_decision)
    print('ENTITES EXTRAITES')
    for texte_e, type_e in entites:
        print(f'  [{type_e}] {texte_e}')

    return pred_label, confidence, entites

# Tests
exemples = [
    'Le prévenu a procédé au détournement de fonds publics à hauteur de 50 000 euros '
    'en falsifiant les documents comptables entre janvier et mars 2023.',
    'Le tribunal retient que le conducteur était sous empire alcoolique avec un taux de 1.8g/L '
    'causant un accident avec blessures graves.',
    'Il est reproché au prévenu d avoir importé des marchandises de contrebande '
    'en dissimulant leur valeur réelle lors du passage en douane.',
]

for i, exemple in enumerate(exemples, 1):
    print(f'\n--- EXEMPLE {i} ---')
    pipeline_complet(exemple)

---
## Bilan v2 — Ce qui a changé

- `pyjudilibre` remplace les appels `requests` manuels — code plus propre et plus robuste
- Accès aux **décisions pénales** (crimes, délits, contraventions) depuis le 31/12/2025
- Endpoint `/export` utilisé pour récupérer des lots de décisions complètes
- Distribution par juridiction ajoutée dans l'exploration
- `target_names` construit dynamiquement depuis les labels présents (plus d'erreur classification_report)

